# 🧬 Notebook 9a — Single-Patient Scoring Module
## Build-order Step 5 | CVD Digital Twin Project | CAD_DT_Final

---

## Purpose
Combines everything built so far — NB1's `build_lifestyle_features()`, NB2's `build_clinical_features()`, NB6's trained pipelines, NB7's fixed-weight PRS integration, NB8's Platt calibrators and SHAP domain attribution — into one function that scores a **single new patient** end to end. This is the ML side only; it does **not** touch Pulse (that's Step 6, `nb9b_pulse_bridge`).

**Output for each patient:** `{ml_risk, risk_band, domain_attribution}`

---

## Verification status (2026-08-03, final)
- NB1, NB2, NB8 all confirmed via real Colab execution, zero errors.
- **Clinical path**: verified against the REAL `clinical_pipeline.pkl` and REAL `cl_calibrator.pkl`. Caught and fixed a real bug during testing — `build_clinical_features()`'s `fitted_scaler` parameter was accepted but never applied, so continuous features were fed to the model in raw, unscaled units. Sanity-tested: high-risk patient → 0.9142 (Very High), low-risk patient → 0.0818 (Low).
- **Lifestyle path**: verified against the REAL `lifestyle_pipeline.pkl` (XGBoost, 14 features) and REAL `ls_calibrator.pkl`. Sanity-tested: high-risk patient → 0.8429 (Very High), low-risk patient → 0.1323 (Low).
- **One remaining gap**: the lifestyle SHAP background used below is rebuilt from hand-built patients (the originally saved background was generated against an earlier synthetic model with the wrong feature count — 5 columns instead of 14). Functionally correct, but not the literal file from your Drive. Swap in the real `shap_background_lifestyle.pkl` in Section 5 if you have it — no code changes needed.

## What this notebook does **NOT** do
- ❌ Touch Pulse physiology simulation — that's Step 6.
- ❌ Silently accept physiologically implausible new-patient input. Continuous clinical values out of NB2's IQR fence are **clipped** (matches training-time behavior); lifestyle bounds are enforced by NB1 as **hard errors** (training removed those rows entirely, so there's no fence to clip to — see NB1 Section 13).
- ❌ Assume a fixed model type for either cohort — the SHAP explainer is chosen based on `type(clf).__name__` at runtime (see Section 4), same pattern as NB8.


---
# Section 1 — Setup & Constants

## What is being done
Imports, and the fixed constants that must match NB7's documented decision (2026-07-26): `W1=0.85`, `W2=0.15` for PRS integration, and NB7's risk bands.

## Why it is needed
These constants are shared across every patient scored by this module — defining them once here, matching NB7 exactly, avoids the kind of silent drift this project has caught before (e.g. NB8's `PRS_CONTRIBUTION` hardcoding fix).


In [1]:
import os
import pickle
import numpy as np
import pandas as pd
import shap

# ── Fixed constants — MUST match NB7's documented decision (2026-07-26) ──────────
W1 = 0.85
W2 = 0.15

# Risk bands — copied verbatim from NB7 Section 5
BANDS      = ['Low', 'Moderate', 'High', 'Very High']
THRESHOLDS = [0.0, 0.25, 0.50, 0.75, 1.01]

TREE_MODEL_TYPES = {
    'XGBClassifier', 'RandomForestClassifier', 'GradientBoostingClassifier',
    'LGBMClassifier', 'ExtraTreesClassifier', 'DecisionTreeClassifier',
}

print('✅ Setup complete')
print(f'  W1={W1}  W2={W2}')
print(f'  Bands: {BANDS}')
print(f'  Thresholds: {THRESHOLDS}')


✅ Setup complete
  W1=0.85  W2=0.15
  Bands: ['Low', 'Moderate', 'High', 'Very High']
  Thresholds: [0.0, 0.25, 0.5, 0.75, 1.01]


---
# Section 2 — Domain Maps

## What is being done
Copied verbatim from NB8 Section 7 (2026-07-27 version) — these map each feature name to a domain (lifestyle / clinical) for SHAP-based domain attribution. Genetic contribution is added separately as a fixed constant (NB7/NB8's established design — PRS is not a per-sample SHAP-tractable feature).


In [2]:
LS_DOMAIN_MAP = {
    'lifestyle': [
        'smoke', 'alco', 'active', 'bmi',
        'smoking', 'alcohol', 'physical_activity',
    ],
    'clinical': [
        'age', 'gender', 'height', 'weight',
        'systolic_bp', 'diastolic_bp',
        'cholesterol_level_1', 'cholesterol_level_2', 'cholesterol_level_3',
        'glucose_level_1', 'glucose_level_2', 'glucose_level_3',
        'ap_hi', 'ap_lo', 'cholesterol', 'gluc',
    ],
}
CL_DOMAIN_MAP = {
    'clinical': [
        'age', 'resting_bp', 'cholesterol', 'max_heart_rate', 'oldpeak',
        'fasting_blood_sugar',
        'resting_ecg_0.0', 'resting_ecg_1.0', 'resting_ecg_2.0',
        'sex', 'trestbps', 'chol', 'thalach', 'ca', 'thal',
    ],
    'lifestyle': [],
}
print('✅ Domain maps loaded')


✅ Domain maps loaded


---
# Section 3 — Single-Patient Feature Builders

## What is being done
`build_lifestyle_features()` and `build_clinical_features()` — copied verbatim from NB1 Section 13 and NB2 Section 14, the single source of truth for how a raw patient dict becomes a model-ready row.

## Why row-deletion becomes row-validation (lifestyle) vs. clipping (clinical)
NB1 enforced its physiological bounds by **deleting** training rows — deletion has no single-row equivalent, so `build_lifestyle_features()` raises an explicit error instead of silently extrapolating into a region with zero training support. NB2 enforced its bounds by **clipping** (capping) — that behavior transfers directly to a single row, so `build_clinical_features()` clips out-of-fence continuous values rather than rejecting them, exactly matching what happened to the training data itself.

## Reindexing to the deployed model's exact feature order
Both functions accept `target_features` — pass `pipeline.calibrated_classifiers_[0].estimator.named_steps['scaler'].feature_names_in_.tolist()` from the real fitted pipeline (Section 6 does this automatically).


In [3]:
_LS_BOUNDS = {
    'systolic_bp': (70, 250), 'diastolic_bp': (40, 150),
    'height_cm': (100, 220), 'weight_kg': (30, 200), 'bmi': (12, 60),
}


def build_lifestyle_features(raw: dict, target_features=None) -> pd.DataFrame:
    """Build a single-row feature DataFrame for a new patient, matching NB1's
    training-time transformations exactly. Raises ValueError on physiologically
    implausible input (deletion in training has no single-row equivalent)."""
    row = dict(raw)
    rename_map = {
        'height': 'height_cm', 'weight': 'weight_kg',
        'ap_hi': 'systolic_bp', 'ap_lo': 'diastolic_bp',
        'cholesterol': 'cholesterol_level', 'gluc': 'glucose_level',
        'smoke': 'smoking', 'alco': 'alcohol', 'active': 'physical_activity',
    }
    for old, new in rename_map.items():
        if old in row:
            row[new] = row.pop(old)

    required = ['age', 'gender', 'height_cm', 'weight_kg', 'systolic_bp', 'diastolic_bp',
                'cholesterol_level', 'glucose_level', 'smoking', 'alcohol', 'physical_activity']
    missing = [f for f in required if f not in row]
    if missing:
        raise ValueError(f'Missing required raw lifestyle field(s): {missing}')

    if row['gender'] in ('f', 0, '0'):
        row['gender'] = 0
    elif row['gender'] in ('m', 1, '1'):
        row['gender'] = 1
    else:
        raise ValueError(f"gender must be 'f'/'m' (or 0/1) — got {row['gender']!r}")

    for field in ['systolic_bp', 'diastolic_bp', 'height_cm', 'weight_kg']:
        lo, hi = _LS_BOUNDS[field]
        if not (lo <= row[field] <= hi):
            raise ValueError(
                f'{field}={row[field]} is outside the physiologically plausible range '
                f'[{lo}, {hi}] used to filter training data — the model has no training '
                f'support for this value and cannot reliably score it.'
            )
    if row['diastolic_bp'] >= row['systolic_bp']:
        raise ValueError(
            f"diastolic_bp ({row['diastolic_bp']}) >= systolic_bp ({row['systolic_bp']}) "
            f"is physiologically impossible."
        )

    height_m = row['height_cm'] / 100
    bmi = row['weight_kg'] / (height_m ** 2)
    lo, hi = _LS_BOUNDS['bmi']
    if not (lo <= bmi <= hi):
        raise ValueError(
            f'Derived BMI={bmi:.1f} is outside the physiologically plausible range '
            f'[{lo}, {hi}] used to filter training data — the model has no training '
            f'support for this value.'
        )
    row['bmi'] = bmi
    for f in ['height_cm', 'weight_kg']:
        row.pop(f, None)

    for col, val in [('cholesterol_level', row.pop('cholesterol_level')),
                      ('glucose_level', row.pop('glucose_level'))]:
        if int(val) not in (1, 2, 3):
            raise ValueError(f'{col} must be 1, 2, or 3 — got {val!r}')
        for level in (1, 2, 3):
            row[f'{col}_{level}'] = 1 if int(val) == level else 0

    df_row = pd.DataFrame([row])
    if target_features is not None:
        missing_cols = set(target_features) - set(df_row.columns)
        if missing_cols:
            raise ValueError(f'Built lifestyle row is missing expected feature(s): {missing_cols}')
        df_row = df_row.reindex(columns=target_features)
    return df_row


print('✅ build_lifestyle_features() defined')


✅ build_lifestyle_features() defined


In [4]:
def build_clinical_features(raw: dict, fitted_imputer, fitted_scaler, fitted_iqr_bounds,
                             target_features=None) -> pd.DataFrame:
    """Build a single-row feature DataFrame for a new patient, matching NB2's
    training-time transformations exactly. Continuous out-of-fence values are
    CLIPPED (matches NB2 Section 7b's own training-time behavior); categorical
    values outside the known set raise a ValueError (no dummy column exists)."""
    row = dict(raw)
    rename_map = {
        'chest pain type': 'chest_pain_type', 'resting bp s': 'resting_bp',
        'fasting blood sugar': 'fasting_blood_sugar', 'resting ecg': 'resting_ecg',
        'max heart rate': 'max_heart_rate', 'exercise angina': 'exercise_angina',
        'ST slope': 'st_slope',
    }
    for old, new in rename_map.items():
        if old in row:
            row[new] = row.pop(old)

    required = ['age', 'sex', 'chest_pain_type', 'resting_bp', 'cholesterol',
                'fasting_blood_sugar', 'resting_ecg', 'max_heart_rate',
                'exercise_angina', 'oldpeak', 'st_slope']
    missing_fields = [f for f in required if f not in row]
    if missing_fields:
        raise ValueError(f'Missing required raw clinical field(s): {missing_fields}')

    for field, valid in {'sex': [0, 1], 'exercise_angina': [0, 1], 'fasting_blood_sugar': [0, 1]}.items():
        if int(row[field]) not in valid:
            raise ValueError(f'{field}={row[field]!r} not in valid set {valid}')
    ohe_categories = {'chest_pain_type': [1, 2, 3, 4], 'resting_ecg': [0, 1, 2], 'st_slope': [1, 2, 3]}
    for field, valid in ohe_categories.items():
        allowed = valid + ([0] if field == 'st_slope' else [])
        if int(row[field]) not in allowed:
            raise ValueError(f'{field}={row[field]!r} not in valid set {allowed}')

    for col in ['cholesterol', 'resting_bp', 'st_slope']:
        if row[col] == 0:
            row[col] = np.nan

    for col in ['cholesterol', 'resting_bp', 'max_heart_rate', 'oldpeak']:
        if pd.isna(row[col]):
            continue
        lo, hi = fitted_iqr_bounds[col]
        if not (lo <= row[col] <= hi):
            row[col] = min(max(row[col], lo), hi)

    impute_input_cols = fitted_imputer.feature_names_in_.tolist()
    df_impute_in = pd.DataFrame([{c: row[c] for c in impute_input_cols}])[impute_input_cols]
    imputed_arr = fitted_imputer.transform(df_impute_in)
    df_imputed = pd.DataFrame(imputed_arr, columns=impute_input_cols)

    for col in ['cholesterol', 'resting_bp', 'max_heart_rate', 'oldpeak']:
        lo, hi = fitted_iqr_bounds[col]
        df_imputed[col] = df_imputed[col].clip(lo, hi)
    for col in ['cholesterol', 'resting_bp', 'st_slope']:
        df_imputed[col] = df_imputed[col].round().astype(int)

    row = df_imputed.iloc[0].to_dict()

    for field, categories in ohe_categories.items():
        raw_val = row.pop(field)
        if field == 'st_slope':
            val = int(raw_val)
            for cat in categories:
                row[f'{field}_{cat}'] = 1 if val == cat else 0
        else:
            val = float(raw_val)
            for cat in categories:
                row[f'{field}_{float(cat)}'] = 1 if val == float(cat) else 0

    df_row = pd.DataFrame([row])

    # BUG FIX (found 2026-08-03 via required sanity testing): fitted_scaler was
    # accepted as a parameter but never applied — continuous features were fed to
    # clinical_pipeline in raw units instead of NB2's z-scored units, saturating
    # the logistic regression to near-0/near-1 regardless of the actual patient.
    scale_cols = fitted_scaler.feature_names_in_.tolist()
    df_row[scale_cols] = fitted_scaler.transform(df_row[scale_cols])

    if target_features is not None:
        missing_cols = set(target_features) - set(df_row.columns)
        if missing_cols:
            raise ValueError(f'Built clinical row is missing expected feature(s): {missing_cols}')
        df_row = df_row.reindex(columns=target_features)
    return df_row


print('✅ build_clinical_features() defined')


✅ build_clinical_features() defined


---
# Section 4 — SHAP Explainer, Domain Attribution, Risk Banding

## What is being done
`make_explainer_and_values()` mirrors NB8's dispatcher exactly — auto-selects `TreeExplainer` / `LinearExplainer` / generic `shap.Explainer` based on `type(clf).__name__`, never assumes a model type. `domain_attribution_single()` is NB8's `compute_domain_attribution()` collapsed to one patient. `assign_band()` matches NB7's exact thresholds.


In [5]:
def make_explainer_and_values(clf, X_background):
    model_type = type(clf).__name__
    if model_type in TREE_MODEL_TYPES:
        explainer = shap.TreeExplainer(clf)
        shap_vals = explainer.shap_values(X_background)
        if isinstance(shap_vals, list):
            shap_vals = shap_vals[1]
        ev = explainer.expected_value
        expected_value = float(ev[1]) if isinstance(ev, (list, np.ndarray)) else float(ev)
        used = 'TreeExplainer'
    elif hasattr(clf, 'coef_'):
        explainer = shap.LinearExplainer(clf, X_background)
        shap_vals = explainer.shap_values(X_background)
        ev = explainer.expected_value
        expected_value = float(ev[0]) if isinstance(ev, (list, np.ndarray)) else float(ev)
        used = 'LinearExplainer'
    else:
        f = lambda X: clf.predict_proba(X)[:, 1]
        explainer = shap.Explainer(f, X_background)
        exp = explainer(X_background)
        shap_vals = exp.values
        expected_value = float(np.mean(exp.base_values))
        used = 'shap.Explainer (generic)'
    return shap_vals, expected_value, used


def domain_attribution_single(shap_row, feature_names, domain_map, prs_contribution):
    row_shap = dict(zip(feature_names, np.abs(shap_row)))
    domain_sums = {}
    for domain, feats in domain_map.items():
        domain_sums[domain] = sum(row_shap.get(f, 0.0) for f in feats)
    domain_sums['genetic'] = prs_contribution

    assigned_feats = {f for feats in domain_map.values() for f in feats}
    unmatched = {k for k in row_shap if k not in assigned_feats}
    unassigned = sum(v for k, v in row_shap.items() if k not in assigned_feats)
    domain_sums['clinical'] = domain_sums.get('clinical', 0.0) + unassigned

    total = sum(domain_sums.values())
    pct = {k: float(v / total * 100 if total > 0 else 0.0) for k, v in domain_sums.items()}
    return pct, unmatched


def assign_band(p):
    for i in range(len(THRESHOLDS) - 1):
        if THRESHOLDS[i] <= p < THRESHOLDS[i + 1]:
            return BANDS[i]
    return BANDS[-1]


print('✅ SHAP / domain attribution / band-assignment functions defined')


✅ SHAP / domain attribution / band-assignment functions defined


---
# Section 5 — Model Bundle

## What is being done
`PatientScoringModel` loads every artifact once (pipelines, calibrators, PRS CSV, SHAP backgrounds, NB2's imputer/scaler/iqr_bounds) and exposes `score_lifestyle()` / `score_clinical()` — each computes `p_base → p_integrated → p_calibrated`, runs SHAP, and returns the domain attribution, all reusing the exact fitted objects from your real pipeline.

## Why it is structured this way
Loading everything once at construction (not per-call) matters for a web backend calling this repeatedly — re-loading pickles and re-fitting nothing per request keeps latency low.


In [6]:
class PatientScoringModel:
    def __init__(self, model_dir, explainability_dir, genetics_dir):
        with open(os.path.join(model_dir, 'lifestyle_pipeline.pkl'), 'rb') as f:
            self.lifestyle_pipeline = pickle.load(f)
        with open(os.path.join(model_dir, 'clinical_pipeline.pkl'), 'rb') as f:
            self.clinical_pipeline = pickle.load(f)
        with open(os.path.join(model_dir, 'ls_calibrator.pkl'), 'rb') as f:
            self.ls_calibrator = pickle.load(f)
        with open(os.path.join(model_dir, 'cl_calibrator.pkl'), 'rb') as f:
            self.cl_calibrator = pickle.load(f)

        prs_df = pd.read_csv(os.path.join(genetics_dir, 'prs_population_score.csv'))
        assert 'sigmoid_prs' in prs_df.columns, "PRS CSV missing sigmoid_prs — check NB4 version"
        self.sigmoid_prs = float(prs_df['sigmoid_prs'].values[0])

        with open(os.path.join(explainability_dir, 'shap_background_lifestyle.pkl'), 'rb') as f:
            self.ls_bg = pickle.load(f)
        with open(os.path.join(explainability_dir, 'shap_background_clinical.pkl'), 'rb') as f:
            self.cl_bg = pickle.load(f)

        self._ls_inner = self.lifestyle_pipeline.calibrated_classifiers_[0].estimator
        self._cl_inner = self.clinical_pipeline.calibrated_classifiers_[0].estimator
        self.ls_scaler = self._ls_inner.named_steps['scaler']
        self.cl_scaler = self._cl_inner.named_steps['scaler']
        self.ls_clf = self._ls_inner.named_steps['clf']
        self.cl_clf = self._cl_inner.named_steps['clf']
        self.ls_features = self.ls_scaler.feature_names_in_.tolist()
        self.cl_features = self.cl_scaler.feature_names_in_.tolist()

        clinical_dir = model_dir.replace('Models', 'Clinical')
        with open(os.path.join(clinical_dir, 'clinical_imputer.pkl'), 'rb') as f:
            self.cl_imputer = pickle.load(f)
        with open(os.path.join(clinical_dir, 'clinical_scaler.pkl'), 'rb') as f:
            self.cl_prep_scaler = pickle.load(f)
        with open(os.path.join(clinical_dir, 'clinical_iqr_bounds.pkl'), 'rb') as f:
            self.cl_iqr_bounds = pickle.load(f)

        print('✅ PatientScoringModel loaded:')
        print(f'   Lifestyle: {type(self.ls_clf).__name__}  ({len(self.ls_features)} features)')
        print(f'   Clinical : {type(self.cl_clf).__name__}  ({len(self.cl_features)} features)')
        print(f'   sigmoid_prs: {self.sigmoid_prs:.6f}')

    def score_lifestyle(self, raw_patient: dict) -> dict:
        X = build_lifestyle_features(raw_patient, target_features=self.ls_features)
        p_base = float(self.lifestyle_pipeline.predict_proba(X)[:, 1][0])
        p_integrated = float(np.clip(W1 * p_base + W2 * self.sigmoid_prs, 0, 1))
        p_calibrated = float(self.ls_calibrator.predict_proba([[p_integrated]])[:, 1][0])

        X_scaled = pd.DataFrame(self.ls_scaler.transform(X), columns=self.ls_features)
        bg = self.ls_bg['background']
        shap_vals, _, explainer_used = make_explainer_and_values(
            self.ls_clf, pd.concat([bg, X_scaled], ignore_index=True)
        )
        this_row_shap = shap_vals[-1]
        domain_pct, unmatched = domain_attribution_single(
            this_row_shap, self.ls_features, LS_DOMAIN_MAP, W2 * self.sigmoid_prs
        )
        return {
            'cohort': 'lifestyle',
            'p_base': p_base, 'p_integrated': p_integrated, 'ml_risk': p_calibrated,
            'risk_band': assign_band(p_calibrated),
            'domain_attribution': domain_pct,
            '_explainer_used': explainer_used,
            '_unmatched_features': sorted(unmatched) if unmatched else [],
        }

    def score_clinical(self, raw_patient: dict) -> dict:
        X = build_clinical_features(
            raw_patient, self.cl_imputer, self.cl_prep_scaler, self.cl_iqr_bounds,
            target_features=self.cl_features
        )
        p_base = float(self.clinical_pipeline.predict_proba(X)[:, 1][0])
        p_integrated = float(np.clip(W1 * p_base + W2 * self.sigmoid_prs, 0, 1))
        p_calibrated = float(self.cl_calibrator.predict_proba([[p_integrated]])[:, 1][0])

        X_scaled = pd.DataFrame(self.cl_scaler.transform(X), columns=self.cl_features)
        bg = self.cl_bg['background']
        shap_vals, _, explainer_used = make_explainer_and_values(
            self.cl_clf, pd.concat([bg, X_scaled], ignore_index=True)
        )
        this_row_shap = shap_vals[-1]
        domain_pct, unmatched = domain_attribution_single(
            this_row_shap, self.cl_features, CL_DOMAIN_MAP, W2 * self.sigmoid_prs
        )
        return {
            'cohort': 'clinical',
            'p_base': p_base, 'p_integrated': p_integrated, 'ml_risk': p_calibrated,
            'risk_band': assign_band(p_calibrated),
            'domain_attribution': domain_pct,
            '_explainer_used': explainer_used,
            '_unmatched_features': sorted(unmatched) if unmatched else [],
        }


print('✅ PatientScoringModel class defined')


✅ PatientScoringModel class defined


---
# Section 6 — Load Real Artifacts & Test on Hand-Built Patients

## What is being done
Loads the model bundle from your real Drive paths, then scores four hand-built patients (high/low risk × lifestyle/clinical) exactly as required before touching Pulse — confirms the numbers look sane against NB7/NB8's batch results.

**Update `BASE_DIR` below if your folder structure differs.**


In [7]:
BASE_DIR = "/content/drive/MyDrive/CAD_DT_Final/"

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass  # not running in Colab

MODEL_DIR = os.path.join(BASE_DIR, "Outputs", "Models")
EXPL_DIR  = os.path.join(BASE_DIR, "Outputs", "Explainability")
GEN_DIR   = os.path.join(BASE_DIR, "Outputs", "Genetics")

model = PatientScoringModel(model_dir=MODEL_DIR, explainability_dir=EXPL_DIR, genetics_dir=GEN_DIR)


Mounted at /content/drive
✅ PatientScoringModel loaded:
   Lifestyle: XGBClassifier  (14 features)
   Clinical : LogisticRegression  (10 features)
   sigmoid_prs: 0.502258


In [8]:
import json

test_patients = {
    "Clinical - high risk": ('clinical', {
        'age': 58, 'sex': 1, 'chest pain type': 4, 'resting bp s': 145, 'cholesterol': 260,
        'fasting blood sugar': 1, 'resting ecg': 1, 'max heart rate': 120,
        'exercise angina': 1, 'oldpeak': 2.3, 'ST slope': 2,
    }),
    "Clinical - low risk": ('clinical', {
        'age': 35, 'sex': 0, 'chest pain type': 1, 'resting bp s': 110, 'cholesterol': 180,
        'fasting blood sugar': 0, 'resting ecg': 0, 'max heart rate': 175,
        'exercise angina': 0, 'oldpeak': 0.0, 'ST slope': 1,
    }),
    "Lifestyle - high risk": ('lifestyle', {
        'age': 60, 'gender': 'm', 'height': 175, 'weight': 100,
        'ap_hi': 155, 'ap_lo': 98, 'cholesterol': 3, 'gluc': 3,
        'smoke': 1, 'alco': 1, 'active': 0,
    }),
    "Lifestyle - low risk": ('lifestyle', {
        'age': 28, 'gender': 'f', 'height': 165, 'weight': 58,
        'ap_hi': 110, 'ap_lo': 70, 'cholesterol': 1, 'gluc': 1,
        'smoke': 0, 'alco': 0, 'active': 1,
    }),
}

print('=' * 60)
print('  SECTION 6: Sanity test — hand-built patients')
print('=' * 60)
for label, (cohort, raw) in test_patients.items():
    result = model.score_clinical(raw) if cohort == 'clinical' else model.score_lifestyle(raw)
    print(f'\n--- {label} ---')
    print(f"  p_base={result['p_base']:.4f}  p_integrated={result['p_integrated']:.4f}  "
          f"ml_risk={result['ml_risk']:.4f}  band={result['risk_band']}")
    print(f"  explainer={result['_explainer_used']}  domain: " +
          ', '.join(f'{k}={v:.1f}%' for k, v in result['domain_attribution'].items()))
    if result['_unmatched_features']:
        print(f"  ⚠️  Unmatched features (check domain maps): {result['_unmatched_features']}")

print('\n[SECTION 6 COMPLETE] ✅')


  SECTION 6: Sanity test — hand-built patients

--- Clinical - high risk ---
  p_base=0.9593  p_integrated=0.8908  ml_risk=0.9142  band=Very High
  explainer=LinearExplainer  domain: clinical=97.3%, lifestyle=0.0%, genetic=2.7%

--- Clinical - low risk ---
  p_base=0.0470  p_integrated=0.1153  ml_risk=0.0818  band=Low
  explainer=LinearExplainer  domain: clinical=97.9%, lifestyle=0.0%, genetic=2.1%

--- Lifestyle - high risk ---
  p_base=0.8525  p_integrated=0.8000  ml_risk=0.8429  band=Very High
  explainer=TreeExplainer  domain: lifestyle=15.3%, clinical=81.6%, genetic=3.1%

--- Lifestyle - low risk ---
  p_base=0.1031  p_integrated=0.1630  ml_risk=0.1323  band=Low
  explainer=TreeExplainer  domain: lifestyle=7.3%, clinical=90.5%, genetic=2.2%

[SECTION 6 COMPLETE] ✅
